In [1]:
# Import statements
import numpy as np
import os
from pathlib import Path
import csv
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
import scipy.io
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.cluster import AgglomerativeClustering

In [2]:
# load in each time series
basepath = "/Users/rachelfox/FPI_RS_share/PID7"
time_series_dict = {'r13':[], 'veh':[], 'shm':[]}
     
for folder in os.listdir(basepath):
   folder_path = os.path.join(basepath, folder)
   if os.path.isdir(folder_path):
    for file_name in os.listdir(folder_path):
        if file_name.endswith('.nii'):
            nii_path = os.path.join(folder_path, file_name)
            img = nib.load(nii_path)
            data = img.get_fdata()
            time_series = data.reshape(np.prod(data.shape[0:3]), -1)
            print(time_series.shape)
            grp = file_name[-7:-4]
            print(grp)
            time_series_dict[grp].append(np.array(time_series))


(230400, 450)
r13
(230400, 450)
r13
(230400, 450)
shm
(230400, 450)
veh
(230400, 450)
r13
(230400, 450)
r13
(230400, 450)
shm
(230400, 450)
veh
(230400, 450)
shm
(230400, 450)
veh
(230400, 450)
veh
(230400, 450)
r13
(230400, 450)
shm
(230400, 450)
veh
(230400, 450)
r13
(230400, 450)
shm
(230400, 450)
veh
(230400, 450)
veh
(230400, 450)
shm
(230400, 450)
veh
(230400, 450)
shm
(230400, 450)
shm
(230400, 450)
r13
(230400, 450)
veh
(230400, 450)
veh
(230400, 450)
shm
(230400, 450)
r13
(230400, 450)
shm
(230400, 450)
shm


In [ ]:

# individual subject
#for group_name, time_series_list in time_series_dict.items():
#    for i, ts in enumerate(time_series_list):
#        make_clusters(ts.T, f"Cluster for group {group_name}, Subject {i+1}")

# by group
linkage_grps = {}
concat_ts = {}
for group_name, time_series_list in time_series_dict.items():
    print(group_name)
    concatenated_data = np.mean(time_series_list, axis=0) 
    print(concatenated_data.shape)
    scaler = StandardScaler()
    standardized_data = scaler.fit_transform(concatenated_data)
    sample_size = 5000
    downsampled_data = resample(standardized_data, n_samples=sample_size, random_state=42) # since I don't have gpu

    Z = linkage(standardized_data, method='ward', metric='euclidean')
    plt.figure(figsize=(10, 7))
    dendrogram(Z)
    plt.title(group_name)
    plt.show()

    linkage_grps[group_name] = Z
    concat_ts[group_name] = concatenated_data



r13
(230400, 450)


In [ ]:
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import fcluster
from sklearn.utils import resample

# use silhouette score to find max number of clusters
max_clusters = 30

for group_name, time_series_list in time_series_dict.items():
    print(group_name)

    concatenated_data = np.mean(time_series_list, axis=0) 
    print(concatenated_data.shape)

    scaler = StandardScaler()
    standardized_data = scaler.fit_transform(concatenated_data)
    downsampled_data = resample(standardized_data, n_samples=int(0.05 * 230400), random_state=42) # since I don't have gpu
    print(downsampled_data.shape)
    Z = linkage(standardized_data, method='ward', metric='euclidean')
    print("completed linkage")

    silhouette_scores = []
    for k in range(2, max_clusters + 1): 
        labels = fcluster(Z, t=k, criterion='maxclust')
        score = silhouette_score(concatenated_data, labels, metric='euclidean')
        silhouette_scores.append(score)

    optimal_k = range(2, max_clusters + 1)[np.argmax(silhouette_scores)]
    plt.plot(range(2, max_clusters + 1), silhouette_scores, marker='o')
    plt.xlabel('Number of Clusters')
    plt.ylabel('Silhouette Score')
    plt.title('Silhouette Analysis for Optimal Clusters')
    plt.show()

    print(f"Optimal number of clusters: {optimal_k}")


r13
(230400, 450)
(11520, 450)


In [ ]:
# reproject back onto the brain
import numpy as np
from scipy.cluster.hierarchy import fcluster
from sklearn.preprocessing import StandardScaler

# Function to calculate the average time series per cluster
def cluster_average_time_series(data, cluster_labels, num_clusters):
    averaged_time_series = np.zeros((num_clusters, data.shape[1]))
    
    # get mean ts for the cluster
    for cluster_id in range(1, num_clusters + 1):
        cluster_voxels = data[cluster_labels == cluster_id, :]
        averaged_time_series[cluster_id - 1, :] = cluster_voxels.mean(axis=0)
    
    return averaged_time_series


num_clusters = 20 # use optimal number of clusters
original_data = nib.load("/Users/rachelfox/FPI_RS_share/PID7/01fa_PID7/filtered_func_data_01fa_r13.nii").get_fdata()
spatial_shape = original_data.shape

clustered_time_series = {}

# apply cluster to aggregated time series and reproject back onto brain
for group_name, time_series_list in time_series_dict.items():
    hierarchical_cluster = AgglomerativeClustering(n_clusters=num_clusters, affinity='euclidean', linkage='ward')
    data = np.concatenate(time_series_list) 
    cluster_labels = hierarchical_cluster.fit_predict(data)

    # Calculate the average time series for each cluster
    cluster_ts = cluster_average_time_series(data, cluster_labels, num_clusters)
    clustered_time_series[group_name] = cluster_ts

    # Project the gradients onto the brain surface using shape of the nii file
    cluster_map = cluster_labels.reshape(spatial_shape)
    new_img = nib.Nifti1Image(cluster_map.astype(np.int32), affine=img.affine, header=img.header)
    nib.save(new_img, "/Users/rachelfox/FPI_RS_share/PID7/cluster_map_01fa_r13.nii")
